# 3 · Network versus approximation

Load saved outputs without simulating. The configuration fingerprint prevents accidentally comparing different resets, drives, time steps, or connectivity parameters.

## Load matching results

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sc_cortex.config import Config
from sc_cortex.results import save_result, load_result

c = Config.load('config.json')
print(c)
print('Run ID:', c.fingerprint)

network = load_result('network', c)
approximation = load_result('approximation', c)


## Quantitative comparison
These errors describe this run, not statistical equivalence. Full validation requires longer records, multiple independent network seeds, sample-size sensitivity, and a time-step check.

In [ ]:
from sc_cortex.results import compare
metrics = compare(network, approximation)
metrics

## Overlay
Raw spectra are used for errors. The dashed horizontal lines mark the respective mean rates.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4), constrained_layout=True)
for label, data, color in [('SNN', network, '#d16b91'), ('Gaussian approximation', approximation, '#dc8b45')]:
    keep = (data['freq']>0)&(data['freq']<=300)
    ax[0].semilogy(data['freq'][keep], data['psd'][keep], color=color, label=label, alpha=.85)
    ax[0].axhline(float(data['mean_rate']), color=color, ls=':', alpha=.6)
ax[0].set(xlabel='Frequency (Hz)', ylabel='Single-neuron PSD (Hz)'); ax[0].legend()
ax[1].bar(['SNN sample', 'Approximation'], [network['mean_rate'], approximation['mean_rate']], color=['#d16b91','#dc8b45'])
ax[1].set(ylabel='Mean firing rate (Hz)', title=f"Iteration converged: {bool(approximation['converged'])}")
folder = Path('results')/c.fingerprint
fig.savefig(folder/'comparison.png', dpi=180)
plt.show()

## ISI variability
Compare per-neuron and per-trial CVs over matching measurement durations. Short traces can bias CV estimates.

In [ ]:
for label, data, color in [('SNN',network,'#d16b91'), ('Approximation',approximation,'#dc8b45')]:
    cv = data['cv'][np.isfinite(data['cv'])]
    if len(cv):
        plt.hist(cv, bins=20, alpha=.5, color=color, label=f'{label}: mean {cv.mean():.3f}')
plt.xlabel('ISI coefficient of variation'); plt.ylabel('Neurons / trials'); plt.legend(); plt.show()

## Questions for Rodrigo
- Which Dummer figure and reset should be our first quantitative target?
- Are the network activity and its cross-correlations sufficiently asynchronous for this closure?
- Does disagreement shrink with longer runs, finer time steps, and more independent trials?
- Which heterogeneity should we introduce first after validating this baseline?

In [ ]:
# Write your observations here after inspecting the results.
observations = {'agreement': '', 'disagreement': '', 'next_test': ''}
observations